# Projeto ETL - Distribuidora Vetor
## Fase 1 - E : Inicialização e Extração de Dados

In [1]:
# Inicializando todas as bibliotecas necessárias para o ETL (apenas uma vez no começo do notebook).
import pandas as pd      # Primeira vez usando a biblioteca pandas: usaremos métodos de leitura (read_csv), manipulação de DataFrames e operações em lote.
import re                # Primeira vez usando a biblioteca re (Regular Expressions): usaremos re.sub() e re.match() para limpeza de caracteres e validação de padrões.
import unicodedata       # Primeira vez usando a biblioteca unicodedata: usaremos unicodedata.normalize() na fase final para remover acentos dos e-mails.
import requests          # Primeira vez usando a biblioteca requests: usaremos requests.get() na Fase 3 para realizar chamadas HTTP à API do ViaCEP.
import time              # Primeira vez usando a biblioteca time: usaremos time.sleep() na Fase 3 para criar pequenos atrasos e não sobrecarregar a API pública.

# Extensão para recarregar automaticamente funções caso você as separe em arquivos .py no futuro
%load_ext autoreload
%autoreload 2

In [2]:
# Lendo o arquivo original da pasta dados/ do projeto
# Usamos a função pd.read_csv() do pandas. Os argumentos dtype=str e keep_default_na=False garantem que leremos tudo como texto bruto inicial.
df_original = pd.read_csv("../dados/clientes_distribuidora_vetor.csv", sep=";", dtype=str, keep_default_na=False)

# Criamos uma cópia de trabalho para não alterarmos a referência original que usaremos na auditoria final
df = df_original.copy()

# Visualizando as primeiras 5 linhas
df.head()

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926-750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140-010,Brasília,DF,2023-10-07,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140-010,Brasília,DF,28/02/2022,Camila Torres
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929-000,Brasília,DF,"Dec 24, 2023",Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000-000,Brasília,DF,08-01-2022,Diego Martins


### Avaliação Inicial dos Dados

In [3]:
display(df)

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926-750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140-010,Brasília,DF,2023-10-07,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140-010,Brasília,DF,28/02/2022,Camila Torres
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929-000,Brasília,DF,"Dec 24, 2023",Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000-000,Brasília,DF,08-01-2022,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82.278.248/9638-34,+55 61 97118-8177,contato@casadastinta.com.br,Águas Claras,,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83.930.103/1051-83,61999788677,contato@distribuidor.com.br,TAGUATINGA,,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31.165.667/0106-51,(61) 94139-4116,contato@bazarestrela.com.br,V. Pires,72006-400,Brasília,DF,"Aug 18, 2022",Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10.801.326/7736-02,+55 61 91035-7396,contato@supermercado.com.br,Aguas claras,71926-750,Brasília,DF,2023-08-09,Juliana Reis
9,CLI0010,Papelaria Horizonte,34.309.805/0097-88,(61) 91931-9320,contato@papelariahor.com.br,Águas Claras/DF,71950-000,Brasília,DF,03/03/2021,Diego Martins


In [4]:
# df.info() vai mostrar os tipos de dados e quantos "não-nulos" o pandas acha que existem
print("\n--- Informações Gerais (Estado Bruto) ---")
df.info()


--- Informações Gerais (Estado Bruto) ---
<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            47 non-null     str  
 1   razao_social          47 non-null     str  
 2   cnpj                  47 non-null     str  
 3   telefone              47 non-null     str  
 4   email                 47 non-null     str  
 5   bairro_informado      47 non-null     str  
 6   cep                   47 non-null     str  
 7   cidade                47 non-null     str  
 8   uf                    47 non-null     str  
 9   data_cadastro         47 non-null     str  
 10  vendedor_responsavel  47 non-null     str  
dtypes: str(11)
memory usage: 4.2 KB


**Observações do Estado Bruto:** 
* Todas as colunas estão como tipo de texto (str), inclusive as numéricas e datas.
* O `df.info()` aponta que temos 47 registros não nulos em todas as colunas. Isso é um "falso positivo", pois espaços em branco e strings vazias estão sendo contados como dados válidos.
* Há falta de padronização estrutural visível (CNPJ com e sem formatação, telefones irregulares, e CEPs com preenchimento '00000000').

## Fase 2 - T : Tratamento Inicial e Identificação de Nulos Reais

In [5]:
# df.replace(): método do pandas que procura um valor e troca por outro. 
# A regex r'^\s*$' localiza espaços vazios ou nulos disfarçados e os converte para o nulo nativo (pd.NA).
df = df.replace(r'^\s*$', pd.NA, regex=True)

print("\n--- Informações do Dataset (Após forçar Nulos) ---")
df.info()


--- Informações do Dataset (Após forçar Nulos) ---
<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            47 non-null     str  
 1   razao_social          47 non-null     str  
 2   cnpj                  47 non-null     str  
 3   telefone              47 non-null     str  
 4   email                 47 non-null     str  
 5   bairro_informado      46 non-null     str  
 6   cep                   34 non-null     str  
 7   cidade                47 non-null     str  
 8   uf                    47 non-null     str  
 9   data_cadastro         47 non-null     str  
 10  vendedor_responsavel  47 non-null     str  
dtypes: str(11)
memory usage: 4.2 KB


**Resultados do df.info() Pós-Tratamento:**
* Agora os nulos reais apareceram! Notamos que `bairro_informado` e `cep` caíram em contagem de valores válidos.
* O total não bate mais com 47 em todas as colunas, mostrando o estado real de ausência de informações no banco de dados.

## Fase 3: Higienização de CEPs e Integração ViaCEP (Enriquecimento)
**Estratégia Otimizada:** Em vez de tentar limpar nomes de bairros "no escuro", vamos primeiro higienizar a coluna CEP e consultar o ViaCEP. A API nos trará o bairro 100% correto para sobrescrever os erros do usuário, evitando etapas locais desnecessárias.

In [6]:
# 1. Preparação da Coluna CEP (Requisito para o ViaCEP)
def normalizar_cep_auditoria(valor):
    if pd.isna(valor):
        return pd.NA, 'Já era Nulo original'
    
    # Primeira vez usando re.sub() da biblioteca re: substitui tudo que não for dígito (\D) por vazio.
    apenas_numeros = re.sub(r'\D', '', str(valor))
    
    if len(apenas_numeros) == 0:
        return pd.NA, 'Inválido: Sem caracteres numéricos'
        
    if len(apenas_numeros) > 8:
        return pd.NA, f'Inválido: Excesso de dígitos ({len(apenas_numeros)})'
    
    cep_formatado = apenas_numeros.zfill(8)
    
    if cep_formatado == '00000000':
        return pd.NA, 'Inválido: Preenchimento genérico (00000000)'
        
    if len(apenas_numeros) < 8:
        return cep_formatado, 'Válido: Higienizado e zeros à esquerda restaurados'
    
    return cep_formatado, 'Válido: Higienizado (tamanho original correto)'

# df.apply(): método do pandas que aplica a nossa função linha a linha.
# pd.Series(): converte a tupla de retorno em colunas separadas.
df[['cep', 'auditoria_cep']] = df['cep'].apply(lambda x: pd.Series(normalizar_cep_auditoria(x)))

In [7]:
# 2. Integração com a API (Cache em Memória)
ceps_unicos = df['cep'].dropna().unique()
cache_cep = {}

print(f"Buscando {len(ceps_unicos)} CEPs únicos no ViaCEP...\n")

for cep in ceps_unicos:
    cep_str = str(cep)
    url = f"https://viacep.com.br/ws/{cep_str}/json/"
    
    try:
        # Primeira vez usando requests.get() para chamar a API e time.sleep() para não bloquear o servidor.
        resposta = requests.get(url, timeout=5)
        
        if resposta.status_code == 200:
            dados = resposta.json()
            if dados.get('erro'):
                cache_cep[cep_str] = {'bairro_api': pd.NA, 'status': 'Inválido no ViaCEP'}
            else:
                cache_cep[cep_str] = {'bairro_api': dados.get('bairro', ''), 'status': 'Encontrado'}
        else:
            cache_cep[cep_str] = {'bairro_api': pd.NA, 'status': f'Erro HTTP {resposta.status_code}'}
            
    except requests.exceptions.RequestException:
        cache_cep[cep_str] = {'bairro_api': pd.NA, 'status': 'Timeout/Erro de Rede'}
        
    time.sleep(0.1)

# Aplicando os resultados ao DataFrame
def buscar_dados_cache(cep, campo):
    if pd.isna(cep):
        return pd.NA if campo == 'bairro_api' else 'CEP Nulo localmente'
    return cache_cep[str(cep)][campo]

df['bairro_viacep'] = df['cep'].apply(lambda x: buscar_dados_cache(x, 'bairro_api'))
df['status_viacep'] = df['cep'].apply(lambda x: buscar_dados_cache(x, 'status'))

print("Integração concluída!")

Buscando 14 CEPs únicos no ViaCEP...

Integração concluída!


In [8]:
# 3. Guardando a classificação das divergências para o relatório final
def classificar_diferenca(linha):
    inf = str(linha['bairro_informado']).upper().strip()
    via = str(linha['bairro_viacep']).upper().strip()
    
    if linha['status_viacep'] != 'Encontrado' or pd.isna(linha['bairro_viacep']):
        return 'Sem Bairro ViaCEP'
    if inf == via:
        return 'Iguais'
    if ('ÁGUAS CLARAS' in inf and 'ÁGUAS CLARAS' in via) and (('SUL' in inf and 'SUL' in via) or ('NORTE' in inf and 'NORTE' in via)):
        return 'Sintaxe Diferente (Equivalentes)'
    if 'VICENTE PIRES' in inf and 'VICENTE PIRES' in via:
        return 'Sintaxe Diferente (Equivalentes)'
    if inf in via or via in inf:
        return 'Complemento'
    return 'Divergente / Errado'

df['classificacao_bairro'] = df.apply(classificar_diferenca, axis=1)

# ---> NOVO: Backup do estado antes de apagar as colunas <---
# Criamos o df_analise aqui para guardar as informações da API para o relatório final
df_analise = df[['id_cliente', 'bairro_informado', 'bairro_viacep', 'status_viacep', 'classificacao_bairro']].copy()

# 4. A GRANDE SUBSTITUIÇÃO (Regra de Negócio: O ViaCEP vence)
# A) Apaga o CEP se ele for inválido na API
df.loc[df['status_viacep'] == 'Inválido no ViaCEP', 'cep'] = pd.NA
df.loc[df['status_viacep'] == 'Inválido no ViaCEP', 'auditoria_cep'] = 'Inválido: Reprovado no ViaCEP'

# B) Sobrescreve o bairro sujo original pelo bairro limpo da API (onde a API encontrou algo)
mascara = df['bairro_viacep'].notna()
df.loc[mascara, 'bairro_informado'] = df.loc[mascara, 'bairro_viacep']

# Limpeza das colunas temporárias da base principal
df = df.drop(columns=['bairro_viacep', 'status_viacep', 'classificacao_bairro'])
print("Bairros que tinham CEP válido foram corrigidos com a base dos Correios.")

Bairros que tinham CEP válido foram corrigidos com a base dos Correios.


## Fase 4: Normalização e Auditoria (Restante das Colunas)
Nesta fase, aplicamos as regras de negócio para as colunas restantes. **Importante:** A normalização de bairros agora atuará como um "Fallback" inteligente, ou seja, ela só vai limpar os nomes que não foram sobrescritos pelo ViaCEP (clientes que vieram sem CEP no arquivo original).

In [9]:
# Bairros (Fallback local APENAS para quem NÃO foi encontrado no ViaCEP)
def normalizar_bairro_auditoria(valor):
    if pd.isna(valor) or str(valor).strip() == '':
        return pd.NA, 'Já era Nulo ou Vazio'
    
    texto_original = str(valor).strip()
    texto_upper = texto_original.upper()
    bairro_padrao = None
    
    if 'PIRES' in texto_upper: bairro_padrao = 'Vicente Pires'
    elif 'AGUAS' in texto_upper or 'ÁGUAS' in texto_upper:
        if 'NORTE' in texto_upper: bairro_padrao = 'Águas Claras Norte'
        elif 'SUL' in texto_upper: bairro_padrao = 'Águas Claras Sul'
        else: bairro_padrao = 'Águas Claras'
    elif 'GUARA' in texto_upper or 'GUARÁ' in texto_upper:
        if 'II' in texto_upper: bairro_padrao = 'Guará II'
        elif 'I' in texto_upper: bairro_padrao = 'Guará I'
        else: bairro_padrao = 'Guará'
    elif 'TAGUATINGA' in texto_upper:
        if 'SUL' in texto_upper: bairro_padrao = 'Taguatinga Sul'
        else: bairro_padrao = 'Taguatinga'
        
    if bairro_padrao is None: return texto_original, 'Sem regra: Original mantido'
    if texto_original == bairro_padrao: return bairro_padrao, 'Válido: Já no padrão'
    return bairro_padrao, f'Canonizado: De "{texto_original}" para "{bairro_padrao}"'

# 1. Criamos a coluna de auditoria vazia
df['auditoria_bairro'] = pd.NA

# 2. Descobrimos quem NÃO foi encontrado no ViaCEP (precisa de fallback)
mascara_fallback = df_analise['status_viacep'] != 'Encontrado'

# 3. Aplicamos o fallback SÓ nessas linhas problemáticas
resultados_fallback = df.loc[mascara_fallback, 'bairro_informado'].apply(lambda x: pd.Series(normalizar_bairro_auditoria(x)))

if not resultados_fallback.empty:
    df.loc[mascara_fallback, 'bairro_informado'] = resultados_fallback[0]
    df.loc[mascara_fallback, 'auditoria_bairro'] = resultados_fallback[1]

# 4. Para os bairros que VIERAM do ViaCEP, blindamos a grafia original da API
df.loc[~mascara_fallback, 'auditoria_bairro'] = 'Válido: Trazido do ViaCEP'

# ==========================================
# RESTANTE DAS COLUNAS (CNPJ, Tel, Email, Data)
# ==========================================

# CNPJ
def normalizar_cnpj_auditoria(valor):
    if pd.isna(valor) or str(valor).strip() == '': return pd.NA, 'Nulo ou Vazio'
    numeros = re.sub(r'\D', '', str(valor))
    if len(numeros) == 14:
        return f"{numeros[:2]}.{numeros[2:5]}.{numeros[5:8]}/{numeros[8:12]}-{numeros[12:]}", 'Válido: Máscara aplicada'
    return valor, f'Inválido: {len(numeros)} dígitos numéricos'

df[['cnpj', 'auditoria_cnpj']] = df['cnpj'].apply(lambda x: pd.Series(normalizar_cnpj_auditoria(x)))

# Telefone
def normalizar_telefone_auditoria(valor):
    if pd.isna(valor) or str(valor).strip() == '': return pd.NA, 'Nulo ou Vazio'
    numeros = re.sub(r'\D', '', str(valor))
    extra = ""
    if numeros.startswith('55') and len(numeros) in [12, 13]:
        numeros = numeros[2:]
        extra = " (+55 removido)"
    if len(numeros) == 11: return f"{numeros[:2]} {numeros[2:7]}-{numeros[7:]}", f'Celular aplicado{extra}'
    elif len(numeros) == 10: return f"{numeros[:2]} {numeros[2:6]}-{numeros[6:]}", f'Fixo aplicado{extra}'
    return numeros, f'Inválido: Tamanho {len(numeros)}'

df[['telefone', 'auditoria_telefone']] = df['telefone'].apply(lambda x: pd.Series(normalizar_telefone_auditoria(x)))

# E-mail 
def normalizar_email_auditoria(valor):
    if pd.isna(valor) or str(valor).strip() == '': return pd.NA, 'Nulo'
    texto_original = str(valor).strip()
    texto_limpo = texto_original.lower()
    
    texto_sem_acento = unicodedata.normalize('NFKD', texto_limpo).encode('ASCII', 'ignore').decode('utf-8')
    
    if '@' not in texto_sem_acento or '.' not in texto_sem_acento:
        return texto_sem_acento, 'Inválido: Falta @ ou ponto'
    if str(valor) != texto_sem_acento:
        return texto_sem_acento, 'Válido: Higienizado'
    return texto_sem_acento, 'Válido: Padrão correto'

df[['email', 'auditoria_email']] = df['email'].apply(lambda x: pd.Series(normalizar_email_auditoria(x)))

# Data de Cadastro
def normalizar_data_auditoria(valor):
    if pd.isna(valor) or str(valor).strip() == '': return pd.NaT, 'Nulo'
    texto = str(valor).strip()
    try:
        if re.match(r'^\d{4}-', texto):
            data = pd.to_datetime(texto)
        else:
            data = pd.to_datetime(texto, dayfirst=True)
        return data, 'Convertido (YYYY-MM-DD)'
    except:
        return pd.NaT, f'Inválido: "{valor}"'

df[['data_cadastro', 'auditoria_data_cadastro']] = df['data_cadastro'].apply(lambda x: pd.Series(normalizar_data_auditoria(x)))

## Fase 5: Validação Final e Relatórios

In [10]:
print("\n--- Informações Finais (Pós-ETL) ---")
# Agora vemos os tipos corretos (ex: data_cadastro como datetime64) e as contagens reais de preenchimento
df.info()


--- Informações Finais (Pós-ETL) ---
<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id_cliente               47 non-null     str           
 1   razao_social             47 non-null     str           
 2   cnpj                     47 non-null     str           
 3   telefone                 47 non-null     str           
 4   email                    47 non-null     str           
 5   bairro_informado         46 non-null     str           
 6   cep                      28 non-null     str           
 7   cidade                   47 non-null     str           
 8   uf                       47 non-null     str           
 9   data_cadastro            47 non-null     datetime64[us]
 10  vendedor_responsavel     47 non-null     str           
 11  auditoria_cep            47 non-null     str           
 12  auditoria_b

In [11]:
# Relatório Visual 1: Original vs Final
# Não é necessário usar pd.read_csv novamente, pois df_original já está na memória (Célula 3)

df_visaofinal = pd.DataFrame()
df_visaofinal['id_cliente'] = df['id_cliente']

# O bairro CRU, exatamente como estava no CSV original
df_visaofinal['bairro_antigo_csv'] = df_original['bairro_informado']

# O bairro que ficou definitivamente na sua base principal (pós-ViaCEP e pós-Fallback)
df_visaofinal['bairro_final'] = df['bairro_informado']

# Verifica se o bairro estava errado (Baseado na classificação salva no df_analise)
df_visaofinal['bairro_estava_errado'] = df_analise['classificacao_bairro'].apply(
    lambda x: 'Sim' if x == 'Divergente / Errado' else 'Não'
)

# Verifica se o CEP foi confirmado no ViaCEP (buscando de df_analise)
df_visaofinal['cep_confirmado'] = df_analise['status_viacep'].apply(
    lambda x: 'Sim' if x == 'Encontrado' else f'Não ({x})'
)

pd.set_option('display.max_rows', None)
print("\n--- Auditoria Final: Bairro Original vs Bairro Definitivo ---")
display(df_visaofinal)
pd.reset_option('display.max_rows')


--- Auditoria Final: Bairro Original vs Bairro Definitivo ---


,id_cliente,bairro_antigo_csv,bairro_final,bairro_estava_errado,cep_confirmado
0,CLI0001,Sul (Águas Claras),Sul (Águas Claras),Não,Sim
1,CLI0002,Sul (Aguas Claras),Taguatinga Norte,Sim,Sim
2,CLI0003,Taguatinga - DF,Taguatinga Norte,Sim,Sim
3,CLI0004,aguas claras,Sul (Águas Claras),Sim,Sim
4,CLI0005,Taguatinga/DF,Taguatinga,Não,Não (CEP Nulo localmente)
5,CLI0006,Águas Claras,Águas Claras,Não,Não (CEP Nulo localmente)
6,CLI0007,TAGUATINGA,Taguatinga,Não,Não (CEP Nulo localmente)
7,CLI0008,V. Pires,Setor Habitacional Vicente Pires,Sim,Sim
8,CLI0009,Aguas claras,Sul (Águas Claras),Sim,Sim
9,CLI0010,Águas Claras/DF,Areal,Sim,Sim
